In [ ]:
# plot_counts_per_distance.ipynb

# Cell 01 - Plot a graph of distance vs. measured counters per minute (CPM)

%matplotlib widget

import os

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

datafile_name = "counts_per_distance.csv"


def fit_linear(vec_x, vec_y):
    vec_x = vec_x.reshape(-1, 1)
    model = LinearRegression().fit(vec_x, vec_y)
    m = model.coef_
    b = model.intercept_
    score = model.score(vec_x, vec_y)
    return m, b, score


def fit_quadratic(vec_x, vec_y):
    vec_x = vec_x.reshape(-1, 1)
    transformer = PolynomialFeatures(degree=2, include_bias=False)
    transformer.fit(vec_x)
    vec_x2 = transformer.transform(vec_x)
    model = LinearRegression().fit(vec_x2, vec_y)
    b, a = model.coef_
    c = model.intercept_
    score = model.score(vec_x2, vec_y)
    return a, b, c, score


# Set size of output image
fig = plt.figure()
fig.suptitle("Thorium Mantle Experiment")
ax = plt.gca()

# Read in the data file of counts
folder_path = os.path.dirname(os.path.realpath("__file__"))
file_path = os.path.join(folder_path, datafile_name)
vec_y = np.genfromtxt(f"{file_path}", delimiter=",", skip_header=1)

# Reverse the order of counts as distance was decreasing
vec_y = vec_y[::-1]

# Multiply by 8 because each single 1x Lego block row is 8mm closer
vec_x = np.arange(vec_y.size) * 8

# Plot the best linear fit
x = np.linspace(np.min(vec_x), np.max(vec_x), 500)
m, b, score = fit_linear(vec_x, vec_y)
ax.plot(
    x, m * x + b, color="green", linestyle="--", label=f"Linear ($R^2$={score:.4f})"
)

# Plot the best quadratic fit
a, b, c, score = fit_quadratic(vec_x, vec_y)
ax.plot(x, a * x**2 + b * x + c, color="blue", label=f"Quadratic ($R^2$={score:.4f})")

# Plot the measured counts
ax.scatter(vec_x, vec_y, color="red")

# Decorate the graph with proper lables
ax.set_title(f"Decay Counts Per Distance")
ax.set_xlabel("Distance (mm)")
ax.set_ylabel("Decay Count (per minute)")
ax.legend()
ax.set_ylim(0)
ax.xaxis.set_minor_locator(MultipleLocator(2))

# Save the graph as a PNG image file
ax.figure.savefig(f"Decay Counts Per Distance.png")

plt.show()